# Logistic Regression (LR)

In [12]:
import time
import sys
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from sklearn.linear_model import LogisticRegression

# ---- Feedforward Neural Network ----

# Load the dataset
df = pd.read_csv("dataset_packet_multiclass8_n100000.csv", engine="pyarrow", index_col=0)
(X, y) = (df.drop(["label"], axis=1), df["label"])
(X_train, X_test, y_train, y_test) = train_test_split(X, y, train_size=0.8, random_state=0)

# Train the model

lrc = LogisticRegression()
    
start_time = time.process_time()
lrc.fit(X_train, y_train)
training_time = time.process_time() - start_time

# Model statistics
y_pred = lrc.predict(X_test)
report = classification_report(y_test, y_pred, output_dict=True)
report_text = classification_report(y_test, y_pred)
size_kb = sys.getsizeof(pickle.dumps(lrc)) / 1024
param = lrc.get_params()

print(f"{'-'*53}")
print(f"Features         {lrc.n_features_in_}")
print(f"Accuracy         {round(report["accuracy"], 4)}")
print(f"Precision        {round(report["macro avg"]["precision"], 4)}")
print(f"Recall           {round(report["macro avg"]["recall"], 4)}")
print(f"F1-Score         {round(report["macro avg"]["f1-score"], 4)}")
print(f"Training Time    {round(training_time, 4)} s")
print(f"Size in KB       {round(size_kb, 4)} KB")
print(f"Iteration        {lrc.n_iter_}")

print(f"{'-'*53}")
print(f"Full Report\n{report_text}")

print(f"{'-'*53}")
print(f"Parameters:")
for p, v in param.items(): 
    print(f"{p:30}{v}")

-----------------------------------------------------
Features         128
Accuracy         0.748
Precision        0.7743
Recall           0.7474
F1-Score         0.755
Training Time    31.7188 s
Size in KB       11.0537 KB
Iteration        [100]
-----------------------------------------------------
Full Report
              precision    recall  f1-score   support

           0       0.48      0.61      0.54      2507
           1       0.99      0.93      0.96      2490
           2       0.94      0.77      0.85      2473
           3       0.84      0.92      0.88      2537
           4       0.97      0.96      0.97      2525
           5       0.63      0.60      0.61      2489
           6       0.87      0.61      0.72      2473
           7       0.47      0.57      0.51      2506

    accuracy                           0.75     20000
   macro avg       0.77      0.75      0.75     20000
weighted avg       0.77      0.75      0.76     20000

------------------------------------

C:\Users\danie\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


***
## Model Optimization

***
### Random Search with Cross Validation

In [2]:
import time
import sys
import pandas as pd
import pickle

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from sklearn.model_selection import RandomizedSearchCV

from scipy.stats import loguniform, uniform

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# Load the dataset
df = pd.read_csv("dataset_packet_multiclass8_n100000.csv", engine="pyarrow", index_col=0)
(X, y) = (df.drop(["label"], axis=1), df["label"])
(X_train, X_test, y_train, y_test) = train_test_split(X, y, train_size=0.8, random_state=0)

dtc = LogisticRegression(
    random_state=0,
    # n_jobs=-1
)

param_distribution = {
    'C': loguniform(1e-4, 1e2), 
    'l1_ratio': uniform(0, 1),
    #'dual': False, 
    #'tol': 0.0001, 
    #'fit_intercept': True, 
    #'intercept_scaling': 1, 
    'class_weight': [None, 'balanced'],
    'solver': ['saga'],
    'max_iter': [100, 200, 500],
    #warm_start=False,
}

lrc_random = RandomizedSearchCV(
    estimator=dtc, 
    param_distributions=param_distribution, 
    n_iter=20, 
    cv=3, 
    verbose=2, # Verbose 3 to see whats going on... 
    random_state=0, 
    scoring='accuracy', 
    # n_jobs=-1, 
    refit=True 
)

start_wall_time = time.perf_counter()
lrc_random.fit(X_train, y_train)
total_wall_time = time.perf_counter() - start_wall_time

best_dtc = lrc_random.best_estimator_

y_pred = best_dtc.predict(X_test)
report = classification_report(y_test, y_pred, output_dict=True)
size_kb = sys.getsizeof(pickle.dumps(best_dtc)) / 1024
training_time = lrc_random.cv_results_['mean_fit_time'][lrc_random.best_index_]

print(f"Accuracy         {round(report["accuracy"], 4)}")
print(f"Precision        {round(report["macro avg"]["precision"], 4)}")
print(f"Recall           {round(report["macro avg"]["recall"], 4)}")
print(f"F1-Score         {round(report["macro avg"]["f1-score"], 4)}")
print(f"Training Time    {round(training_time, 4)} s")
print(f"Size in KB       {round(size_kb, 4)} KB")

print(f"\n[+] Best Parameters: {lrc_random.best_params_}")
print(f"[+] Best Cross-Validation Accuracy: {lrc_random.best_score_:.4f}")

Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END C=0.19628224813442816, class_weight=balanced, l1_ratio=0.8442657485810173, max_iter=200, solver=saga; total time=   8.8s
[CV] END C=0.19628224813442816, class_weight=balanced, l1_ratio=0.8442657485810173, max_iter=200, solver=saga; total time=   8.3s
[CV] END C=0.19628224813442816, class_weight=balanced, l1_ratio=0.8442657485810173, max_iter=200, solver=saga; total time=   7.6s
[CV] END C=0.5512926225087429, class_weight=None, l1_ratio=0.4375872112626925, max_iter=500, solver=saga; total time=  11.6s
[CV] END C=0.5512926225087429, class_weight=None, l1_ratio=0.4375872112626925, max_iter=500, solver=saga; total time=  11.6s
[CV] END C=0.5512926225087429, class_weight=None, l1_ratio=0.4375872112626925, max_iter=500, solver=saga; total time=  17.1s
[CV] END C=0.00021891618132748294, class_weight=None, l1_ratio=0.3834415188257777, max_iter=500, solver=saga; total time=   3.0s
[CV] END C=0.00021891618132748294, class_weig

Accuracy         0.7635
Precision        0.7821
Recall           0.7616
F1-Score         0.7679
Training Time    9.6585 s
Size in KB       11.1025 KB

[+] Best Parameters: {'C': 25.947259726504655, 'class_weight': 'balanced', 'l1_ratio': 0.06022547162926983, 'max_iter': 500, 'solver': 'saga'}
[+] Best Cross-Validation Accuracy: 0.7601

Accuracy         0.7635
Precision        0.7821
Recall           0.7616
F1-Score         0.7679
Training Time    6.1593 s
Size in KB       11.1025 KB

[+] Best Parameters: {'C': 25.947259726504655, 'class_weight': 'balanced', 'l1_ratio': 0.06022547162926983, 'max_iter': 500, 'solver': 'saga'}
[+] Best Cross-Validation Accuracy: 0.7601

Accuracy         0.7882
Precision        0.8113
Recall           0.7877
F1-Score         0.7942
Training Time    82.2885 s
Size in KB       11.1025 KB

[+] Best Parameters: {'C': 25.947259726504655, 'class_weight': 'balanced', 'l1_ratio': 0.06022547162926983, 'max_iter': 500, 'solver': 'saga'}
[+] Best Cross-Validation Accuracy: 0.7848


***
## Final Model Test

In [10]:
import time
import sys
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from sklearn.linear_model import LogisticRegression

# ---- Feedforward Neural Network ----

# Load the dataset
df = pd.read_csv("dataset_packet_multiclass8_n100000.csv", engine="pyarrow", index_col=0)
(X, y) = (df.drop(["label"], axis=1), df["label"])
(X_train, X_test, y_train, y_test) = train_test_split(X, y, train_size=0.8, random_state=0)

# Train the model

lrc = LogisticRegression(
    random_state=0,
    n_jobs=-1,
    C=947259726504655,
    class_weight="balanced",
    # l1_ratio=0.06022547162926983,
    max_iter=500,
    solver="saga"
)
    
start_time = time.process_time()
lrc.fit(X_train, y_train)
training_time = time.process_time() - start_time

# Model statistics
start_time = time.process_time()
y_pred = lrc.predict(X_test)
predict_time = time.process_time() - start_time

report = classification_report(y_test, y_pred, output_dict=True)
report_text = classification_report(y_test, y_pred)
size_kb = sys.getsizeof(pickle.dumps(lrc)) / 1024
param = lrc.get_params()

print(f"Features         {lrc.n_features_in_}")
print(f"Accuracy         {round(report["accuracy"], 4)}")
print(f"Precision        {round(report["macro avg"]["precision"], 4)}")
print(f"Recall           {round(report["macro avg"]["recall"], 4)}")
print(f"F1-Score         {round(report["macro avg"]["f1-score"], 4)}")
print(f"Training Time    {round(training_time, 4)} s")
print(f"Predict Time     {round(predict_time, 4)} s")
print(f"Size in KB       {round(size_kb, 4)} KB")
print(f"Iteration        {lrc.n_iter_}")

print(f"\nFull Report\n{report_text}")

print(f"\nParameters:")
for p, v in param.items(): 
    print(f"{p:30}{v}")

Features         128
Accuracy         0.7945
Precision        0.8159
Recall           0.7941
F1-Score         0.8002
Training Time    179.9688 s
Predict Time     0.0 s
Size in KB       11.0684 KB
Iteration        [500]

Full Report
              precision    recall  f1-score   support

           0       0.57      0.65      0.60      2507
           1       0.99      0.93      0.95      2490
           2       0.95      0.83      0.89      2473
           3       0.89      0.92      0.91      2537
           4       0.99      0.96      0.98      2525
           5       0.69      0.65      0.67      2489
           6       0.91      0.68      0.78      2473
           7       0.55      0.73      0.63      2506

    accuracy                           0.79     20000
   macro avg       0.82      0.79      0.80     20000
weighted avg       0.82      0.79      0.80     20000


Parameters:
C                             947259726504655
class_weight                  balanced
dual               

C:\Users\danie\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(



Features         128
Accuracy         0.7945
Precision        0.8159
Recall           0.7941
F1-Score         0.8002
Training Time    137.5469 s
Size in KB       11.0762 KB
Iteration        [500]

Full Report
              precision    recall  f1-score   support

           0       0.57      0.65      0.60      2507
           1       0.99      0.93      0.95      2490
           2       0.95      0.83      0.89      2473
           3       0.89      0.92      0.91      2537
           4       0.99      0.96      0.98      2525
           5       0.69      0.65      0.67      2489
           6       0.91      0.68      0.78      2473
           7       0.55      0.73      0.63      2506

    accuracy                           0.79     20000
   macro avg       0.82      0.79      0.80     20000
weighted avg       0.82      0.79      0.80     20000

-----------------------------------------------------
Parameters:
C                             947259726504655
class_weight                  balanced
dual                          False
fit_intercept                 True
intercept_scaling             1
l1_ratio                      0.06022547162926983
max_iter                      500
multi_class                   deprecated
n_jobs                        -1
penalty                       l2
random_state                  0
solver                        saga
tol                           0.0001
verbose                       0
warm_start                    False
